# Unidad IV – Análisis no supervisado
## Descubrimiento de perfiles de estudiantes con K-Means y PCA

| | |
|---|---|
| **Alumna** | Brisa Adriana Mendoza Vigueras · 10IDGS-G1 |
| **Proyecto** | Análisis del rendimiento académico de estudiantes |
| **Datos** | `data/processed/student_por_modelo.csv` (634 alumnos, sin abandonos) |

En el análisis **supervisado** (Unidad III) le dimos al modelo la respuesta (G3) para que aprendiera a predecirla.
En el análisis **no supervisado** no hay respuesta: el algoritmo busca por sí solo **grupos de alumnos parecidos**
(clustering) y **resume muchas variables en pocas** (PCA).

**Pregunta de esta unidad:** ¿existen perfiles de estudiantes, según sus hábitos y su entorno, y alguno concentra a
los alumnos con mayor riesgo de reprobar?

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

FIG = Path("../docs/figuras")
MODELOS = Path("../modelos")
PROCESSED = Path("../data/processed")
SEMILLA = 42

## Parte A. Preparación

### A.1 Selección de variables

Se eligen **11 variables numéricas** que describen hábitos, vida social, historial y entorno familiar. Son las mismas
propuestas en la Unidad II para el agrupamiento:

| Grupo | Variables | Por qué |
|---|---|---|
| Hábitos de estudio | `studytime`, `absences` | Dedicación y asistencia |
| Historial académico | `failures` | Rezago previo |
| Vida social | `goout`, `freetime`, `Dalc`, `Walc` | Tiempo libre, salidas y consumo de alcohol |
| Entorno familiar | `famrel`, `Medu`, `Fedu` | Relación familiar y escolaridad de los padres |
| Bienestar | `health` | Estado de salud |

**No se usan G1, G2 ni G3:**
- Queremos agrupar a los alumnos por *cómo son*, no por *qué calificación sacaron*.
- Después usamos G3 solo para **interpretar** los grupos: así se ve si un perfil se asocia con el bajo rendimiento
  sin haberlo “forzado”.

Tampoco se usan las variables one-hot de texto (`Mjob_*`, `reason_*`), porque K-Means mide distancias y las columnas
de 0/1 de categorías sin orden distorsionan esas distancias.

In [ ]:
df = pd.read_csv(PROCESSED / "student_por_modelo.csv")
variables = ["studytime", "failures", "absences", "goout", "freetime", "Dalc", "Walc",
             "famrel", "Medu", "Fedu", "health"]
df[variables].describe().T[["mean", "std", "min", "max"]].round(2)

### A.2 Estandarización

K-Means agrupa por **distancia**. Sin estandarizar, `absences` (0 a 15) pesaría mucho más que `studytime` (1 a 4) solo
porque sus números son más grandes. `StandardScaler` deja cada variable con **media 0 y desviación 1**, y así todas
pesan igual.

In [ ]:
escalador = StandardScaler()
Z = escalador.fit_transform(df[variables])
pd.DataFrame(Z, columns=variables).describe().T[["mean", "std"]].round(2).T

## Parte B. Agrupamiento con K-Means

**¿Cómo funciona K-Means?**

1. Se elige el número de grupos *k*.
2. Se colocan *k* centros al azar.
3. Cada alumno se asigna al centro más cercano.
4. Cada centro se mueve al promedio de sus alumnos.
5. Se repiten los pasos 3 y 4 hasta que los grupos ya no cambian.

### B.1 ¿Cuántos clusters? Método del codo y Silhouette Score

- **Inercia (método del codo):** suma de las distancias de cada alumno a su centro. Siempre baja al aumentar *k*; se
  busca el punto donde **deja de bajar mucho** (el “codo”).
- **Silhouette Score:** mide qué tan bien separado está cada alumno de los otros grupos. Va de −1 a 1; más alto es mejor.

In [ ]:
valores_k = range(2, 9)
inercias, siluetas = [], []
for k in valores_k:
    km = KMeans(n_clusters=k, n_init=20, random_state=SEMILLA).fit(Z)
    inercias.append(km.inertia_)
    siluetas.append(silhouette_score(Z, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(list(valores_k), inercias, marker="o", color="#4C72B0")
axes[0].set_title("Método del codo")
axes[0].set_xlabel("Número de clusters (k)")
axes[0].set_ylabel("Inercia")
axes[1].plot(list(valores_k), siluetas, marker="o", color="#DD8452")
axes[1].set_title("Silhouette Score")
axes[1].set_xlabel("Número de clusters (k)")
axes[1].set_ylabel("Silhouette")
for ax in axes:
    ax.axvline(4, color="gray", linestyle="--")
plt.tight_layout()
plt.savefig(FIG / "u4_codo_silhouette.png", dpi=120)
plt.show()

pd.DataFrame({"k": list(valores_k), "inercia": inercias, "silhouette": siluetas}).round(3)

**Elección de k = 4:**

- **El codo no es muy marcado**, pero la inercia deja de bajar con fuerza a partir de 4.
- **El Silhouette es bajo en todos los casos (0.11 a 0.17).** Los perfiles de los alumnos no forman grupos muy
  separados; es normal en datos de cuestionarios, donde las personas varían de forma gradual.
- **k = 2 tiene el Silhouette más alto, pero solo separa en dos bloques muy generales.**
- **k = 4 tiene mejor Silhouette que k = 3 y k = 5** y genera grupos **más fáciles de interpretar y útiles para
  tomar decisiones**, como se ve enseguida.

Se elige **k = 4** como equilibrio entre la calidad estadística y la utilidad para el proyecto.

### B.2 Entrenar el modelo final

In [ ]:
kmeans = KMeans(n_clusters=4, n_init=20, random_state=SEMILLA)
df["cluster"] = kmeans.fit_predict(Z)
round(silhouette_score(Z, df["cluster"]), 3)

### B.3 Características de cada cluster

Se calcula el **promedio de cada variable** en cada grupo, en sus unidades originales. Después se agrega el
rendimiento (promedio de G3 y % de no aprobados), que **no se usó** para formar los grupos.

In [ ]:
perfil = df.groupby("cluster")[variables].mean()
rendimiento = df.groupby("cluster").agg(
    alumnos=("G3", "size"),
    promedio_G3=("G3", "mean"),
    pct_no_aprobados=("aprobado", lambda s: (1 - s).mean() * 100),
)
pd.concat([perfil, rendimiento], axis=1).round(2).T

Para facilitar la lectura, se le da un **nombre a cada cluster** según la característica que más lo distingue. Los
nombres se asignan con reglas sobre los promedios (y no a mano por número), para que el código funcione igual
aunque K-Means numere los grupos en otro orden.

In [ ]:
nombres = {}
nombres[perfil["failures"].idxmax()] = "Historial de reprobación"
nombres[perfil["Walc"].drop(list(nombres)).idxmax()] = "Vida social intensa"
nombres[perfil["Medu"].drop(list(nombres)).idxmax()] = "Entorno favorable"
nombres[perfil.index.difference(list(nombres))[0]] = "Tranquilos con menor apoyo escolar familiar"
df["perfil"] = df["cluster"].map(nombres)

resumen = pd.concat([perfil, rendimiento], axis=1).rename(index=nombres)
resumen.sort_values("pct_no_aprobados")[["alumnos", "promedio_G3", "pct_no_aprobados",
                                         "failures", "studytime", "absences", "goout", "Walc", "Medu", "Fedu"]].round(2)

In [ ]:
z_perfil = pd.DataFrame(Z, columns=variables).groupby(df["perfil"]).mean()
z_perfil = z_perfil.loc[resumen.sort_values("pct_no_aprobados").index]

fig, ax = plt.subplots(figsize=(11, 3.8))
imagen = ax.imshow(z_perfil, cmap="coolwarm", vmin=-1.5, vmax=1.5, aspect="auto")
ax.set_xticks(range(len(variables)), variables, rotation=45, ha="right")
ax.set_yticks(range(len(z_perfil)), z_perfil.index)
for i in range(z_perfil.shape[0]):
    for j in range(z_perfil.shape[1]):
        ax.text(j, i, f"{z_perfil.iloc[i, j]:.1f}", ha="center", va="center", fontsize=8)
fig.colorbar(imagen, ax=ax, label="Desviaciones respecto al promedio general")
ax.set_title("Perfil de cada cluster (rojo = por encima del promedio, azul = por debajo)")
plt.tight_layout()
plt.savefig(FIG / "u4_perfil_clusters.png", dpi=120)
plt.show()

In [ ]:
orden = resumen.sort_values("pct_no_aprobados").index
fig, ax = plt.subplots(figsize=(9, 4.5))
barras = ax.bar(orden, resumen.loc[orden, "pct_no_aprobados"], color=["#55A868", "#8172B3", "#DD8452", "#C44E52"])
ax.bar_label(barras, fmt="%.1f %%")
ax.set_ylim(0, resumen["pct_no_aprobados"].max() * 1.15)
ax.set_title("Porcentaje de alumnos no aprobados en cada perfil")
ax.set_xlabel("Perfil (cluster)")
ax.set_ylabel("% de no aprobados")
plt.xticks(rotation=10)
plt.tight_layout()
plt.savefig(FIG / "u4_no_aprobados_por_cluster.png", dpi=120)
plt.show()

**Descripción de los 4 perfiles** (ordenados de menor a mayor riesgo):

| Perfil | Alumnos | Promedio G3 | % no aprobados | Qué lo caracteriza |
|---|---|---|---|---|
| **Entorno favorable** | 217 (34 %) | 13.46 | **4.1 %** | Padres con mayor escolaridad (Medu 3.5, Fedu 3.3), mejor relación familiar, más tiempo de estudio, casi sin reprobaciones previas y poco alcohol |
| **Tranquilos con menor apoyo escolar familiar** | 237 (37 %) | 12.16 | **11.4 %** | Hábitos parecidos al grupo anterior (buen estudio, pocas faltas, poca vida social), pero con padres de baja escolaridad (Medu 1.8, Fedu 1.5) |
| **Vida social intensa** | 130 (21 %) | 11.25 | **18.5 %** | El más alto en salidas (4.2), alcohol entre semana (2.6) y en fin de semana (4.0), y faltas (5.7); poco tiempo de estudio |
| **Historial de reprobación** | 50 (8 %) | 9.22 | **50.0 %** | Promedio de 1.8 materias reprobadas antes, el menor tiempo de estudio (1.5), muchas faltas y padres con baja escolaridad |

**Diferencias principales entre los grupos:**

- **Riesgo:** el porcentaje de no aprobados va del 4 % al 50 %. Los perfiles se formaron **sin usar la calificación**
  y aun así separan claramente el riesgo. Eso indica que los hábitos y el entorno sí se relacionan con el rendimiento.
- **Entorno familiar:** los dos grupos “tranquilos” tienen hábitos casi iguales, pero el de padres con más estudios
  tiene 1.3 puntos más de promedio y menos de la mitad de no aprobados.
- **Hábitos:** con escolaridad de los padres intermedia, el grupo de vida social intensa tiene el doble de faltas y
  un riesgo mayor que los grupos tranquilos.
- **Historial:** el grupo de reprobación previa es pequeño (8 %), pero **uno de cada dos no aprueba**. Coincide con la
  Unidad III, donde `failures` fue la variable más importante.

## Parte C. Reducción de dimensionalidad con PCA

**PCA (Análisis de Componentes Principales)** combina las 11 variables en nuevas variables llamadas **componentes
principales**, ordenadas de la que más información (varianza) conserva a la que menos. Usando solo las dos primeras
se puede **dibujar a los 634 alumnos en un plano** y ver si los clusters se separan.

### C.1 Varianza explicada

In [ ]:
pca = PCA(random_state=SEMILLA).fit(Z)
varianza = pd.DataFrame({
    "componente": [f"PC{i}" for i in range(1, len(variables) + 1)],
    "varianza_explicada_%": pca.explained_variance_ratio_ * 100,
    "acumulada_%": pca.explained_variance_ratio_.cumsum() * 100,
}).round(1)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(varianza["componente"], varianza["varianza_explicada_%"], color="#4C72B0", label="Individual")
ax.plot(varianza["componente"], varianza["acumulada_%"], color="#C44E52", marker="o", label="Acumulada")
ax.set_title("Varianza explicada por cada componente principal")
ax.set_xlabel("Componente")
ax.set_ylabel("% de varianza")
ax.legend()
plt.tight_layout()
plt.savefig(FIG / "u4_varianza_pca.png", dpi=120)
plt.show()

varianza

**Interpretación:**

- PC1 conserva el **19.6 %** de la información y PC2 el **15.8 %**; juntas suman el **35.4 %**.
- Para conservar el 80 % se necesitan 7 componentes.
- La información está repartida entre muchas variables: no hay una o dos que lo expliquen todo. Esto es coherente
  con el Silhouette bajo.
- Aun así, dos componentes bastan para **visualizar** la estructura general.

### C.2 ¿Qué representa cada componente?

Las **cargas (loadings)** indican cuánto contribuye cada variable original a cada componente.

In [ ]:
cargas = pd.DataFrame(pca.components_[:2].T, index=variables, columns=["PC1", "PC2"]).round(2)
cargas.sort_values("PC1", ascending=False)

- **PC1 = eje de “vida social y alcohol”:** tiene cargas altas en `Walc` (0.55), `Dalc` (0.51), `goout` (0.41) y, en
  menor medida, `freetime` y `absences`. `studytime` carga en negativo (−0.27). Un alumno con PC1 alto sale mucho,
  bebe más y estudia menos.
- **PC2 = eje de “escolaridad de los padres”:** casi todo viene de `Medu` (0.66) y `Fedu` (0.66). Un PC2 alto indica
  padres con más estudios. `failures` carga en negativo (−0.27).

### C.3 Visualización de los clusters en el plano PCA

In [ ]:
componentes = pca.transform(Z)[:, :2]
df["PC1"], df["PC2"] = componentes[:, 0], componentes[:, 1]
colores = {"Entorno favorable": "#55A868", "Tranquilos con menor apoyo escolar familiar": "#8172B3",
           "Vida social intensa": "#DD8452", "Historial de reprobación": "#C44E52"}

fig, ax = plt.subplots(figsize=(10, 7))
for nombre, grupo in df.groupby("perfil"):
    ax.scatter(grupo["PC1"], grupo["PC2"], s=22, alpha=0.6, color=colores[nombre], label=f"{nombre} ({len(grupo)})")
centros = pca.transform(kmeans.cluster_centers_)[:, :2]
ax.scatter(centros[:, 0], centros[:, 1], marker="X", s=250, color="black", label="Centro del cluster")
ax.set_title("Alumnos en el plano de los dos primeros componentes principales")
ax.set_xlabel("PC1 – vida social y alcohol (19.6 %)")
ax.set_ylabel("PC2 – escolaridad de los padres (15.8 %)")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.savefig(FIG / "u4_pca_clusters.png", dpi=120)
plt.show()

**Interpretación del gráfico PCA:**

- **“Vida social intensa”** (naranja) se ubica a la **derecha**: PC1 alto, es decir, salidas y alcohol.
- **“Entorno favorable”** (verde) se ubica **arriba a la izquierda**: padres con más estudios y poca vida social.
- **“Tranquilos con menor apoyo escolar familiar”** (morado) está **abajo a la izquierda**: poca vida social y padres
  con menos estudios.
- **“Historial de reprobación”** (rojo) está **abajo a la derecha**: padres con menos estudios y algo más de faltas y
  salidas. Está mezclado con otros grupos porque lo que más lo distingue es `failures`, y esa variable pesa poco
  en PC1 y PC2.
- Los grupos se **traslapan**, lo que coincide con el Silhouette bajo. Aun así, cada grupo ocupa una zona distinta
  del plano y su posición **tiene sentido** con las variables que lo caracterizan.

## Parte D. Guardar resultados

In [ ]:
joblib.dump({"escalador": escalador, "kmeans": kmeans, "pca": pca, "variables": variables, "nombres": nombres},
            MODELOS / "kmeans_pca_perfiles.joblib")

df[["id_estudiante", "cluster", "perfil", "PC1", "PC2", "G3", "aprobado"] + variables].round(3) \
    .to_csv(PROCESSED / "clusters_estudiantes.csv", index=False, decimal=",")
resumen.round(2).rename_axis("perfil").reset_index().to_csv(PROCESSED / "resumen_clusters.csv", index=False, decimal=",")
df["perfil"].value_counts()

## Parte E. Interpretación y conocimiento obtenido

**Patrones encontrados:**

1. Los estudiantes se organizan en **4 perfiles** a partir de dos ejes principales:
   - la **vida social y el consumo de alcohol** (PC1);
   - la **escolaridad de los padres** (PC2).

   A eso se suma el **historial de reprobación**.
2. El riesgo de no aprobar **no se reparte igual entre los perfiles**: va del 4 % (entorno favorable) al 50 %
   (historial de reprobación).
3. Los grupos se traslapan (Silhouette = 0.13): son **tendencias**, no categorías rígidas. Un alumno puede tener
   rasgos de dos perfiles.

**Relación con las variables originales:**
- El perfil “Historial de reprobación” se define por `failures`, `studytime` y `absences`.
- “Vida social intensa” se define por `goout`, `Dalc` y `Walc`.
- Los dos perfiles “tranquilos” se diferencian por `Medu` y `Fedu`.

**Conocimiento nuevo (que el análisis supervisado no mostraba):**
- **El riesgo se concentra en un grupo pequeño e identificable:** el 8 % de los alumnos (historial de reprobación)
  aporta casi el 30 % de los no aprobados (25 de 85).
- **La vida social intensa es un perfil de riesgo propio**, independiente del historial: casi 1 de cada 5 no aprueba
  aunque la mayoría nunca había reprobado.
- **El entorno familiar marca diferencia aun con buenos hábitos:** a hábitos iguales, los alumnos con padres de
  menor escolaridad tienen casi el triple de no aprobados (11.4 % contra 4.1 %).

**¿Cómo usarlo para tomar decisiones?**
- **Historial de reprobación:** tutoría académica intensiva desde el inicio del ciclo y seguimiento de faltas. Es el
  grupo con mayor prioridad.
- **Vida social intensa:** orientación sobre hábitos, prevención del consumo de alcohol y control de asistencia.
- **Menor apoyo escolar familiar:** asesorías o círculos de estudio que compensen el apoyo que en casa no pueden dar.
- **Entorno favorable:** seguimiento normal; no requiere recursos extra.
- **Combinado con la Unidad III:** el perfil se conoce desde la inscripción. El modelo de alerta temprana con G1
  confirma después qué alumnos de cada perfil necesitan intervención.

**Archivos generados:**
- `modelos/kmeans_pca_perfiles.joblib`
- `data/processed/clusters_estudiantes.csv`
- `data/processed/resumen_clusters.csv`
- Las gráficas `docs/figuras/u4_*.png`